RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\shuvo\AppData\Local\Temp\ipykernel_704\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: book review.pdf
  ✓ Loaded 5 pages

Processing: My Resume (1).pdf
  ✓ Loaded 2 pages

Total documents loaded: 7


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-31T21:09:20+06:00', 'author': 'CPH2637', 'moddate': '2026-07-31T21:09:20+06:00', 'source': '..\\data\\pdf\\book review.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'book review.pdf', 'file_type': 'pdf'}, page_content='Book Review Report \nBook Title - "More Than Meets the Eye: Essays on Bangladeshi Politics" \nBibliographic Information \nAuthor: Ali Riaz \nPublisher: University Press Limited (UPL), Dhaka \nYear of Publication: 2022 \nPages: 238 \nISBN: 9789845063777 \n \n \n1. Introduction : \n \nThe process of political development in  Bangladesh since independence is very complex. The \npolitical trajectory  consists of general election histories , institutional and civilian force and \namendments of the constitution. The book has giv en a concise  analytical insight through some \nthoughtful essays . Mr Riaz critically analyzed that the inner dy

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 7 documents into 18 chunks

Example chunk:
Content: Book Review Report 
Book Title - "More Than Meets the Eye: Essays on Bangladeshi Politics" 
Bibliographic Information 
Author: Ali Riaz 
Publisher: University Press Limited (UPL), Dhaka 
Year of Publi...
Metadata: {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-31T21:09:20+06:00', 'author': 'CPH2637', 'moddate': '2026-07-31T21:09:20+06:00', 'source': '..\\data\\pdf\\book review.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'book review.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-31T21:09:20+06:00', 'author': 'CPH2637', 'moddate': '2026-07-31T21:09:20+06:00', 'source': '..\\data\\pdf\\book review.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'book review.pdf', 'file_type': 'pdf'}, page_content='Book Review Report \nBook Title - "More Than Meets the Eye: Essays on Bangladeshi Politics" \nBibliographic Information \nAuthor: Ali Riaz \nPublisher: University Press Limited (UPL), Dhaka \nYear of Publication: 2022 \nPages: 238 \nISBN: 9789845063777 \n \n \n1. Introduction : \n \nThe process of political development in  Bangladesh since independence is very complex. The \npolitical trajectory  consists of general election histories , institutional and civilian force and \namendments of the constitution. The book has giv en a concise  analytical insight through some \nthoughtful essays . Mr Riaz critically analyzed that the inner dy

embedding And vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Projects\RAG\01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 968.46it/s]


Model loaded successfully. Embedding dimension: 384


VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 36


In [9]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-31T21:09:20+06:00', 'author': 'CPH2637', 'moddate': '2026-07-31T21:09:20+06:00', 'source': '..\\data\\pdf\\book review.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'book review.pdf', 'file_type': 'pdf'}, page_content='Book Review Report \nBook Title - "More Than Meets the Eye: Essays on Bangladeshi Politics" \nBibliographic Information \nAuthor: Ali Riaz \nPublisher: University Press Limited (UPL), Dhaka \nYear of Publication: 2022 \nPages: 238 \nISBN: 9789845063777 \n \n \n1. Introduction : \n \nThe process of political development in  Bangladesh since independence is very complex. The \npolitical trajectory  consists of general election histories , institutional and civilian force and \namendments of the constitution. The book has giv en a concise  analytical insight through some \nthoughtful essays . Mr Riaz critically analyzed that the inner dy

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 18 texts...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.73s/it]


Generated embeddings with shape: (18, 384)
Adding 18 documents to vector store...
Successfully added 18 documents to vector store
Total documents in collection: 54


Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("WordPress Developer and Trainer")

Retrieving documents for query: 'WordPress Developer and Trainer'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_3decf83a_16',
  'content': 'website performance, ﬁxing technical issues, and collaborating with the development team.\nWordPress Developer and Trainer\nMultimedia Academy • Jhenaidah\n01/2019 - 02/2022\nWorked as a WordPress Developer and Trainer at Multimedia Academy, where I developed and maintained WordPress\nwebsites, customized themes and plugins, and conducted training sessions on WordPress and web development. I also\nprovided technical support and guided students in building real-world projects.\nLanguages\nShuvo Biswas\nshuvobs60@gmail.com\n+88 01718449354\nJhenaidha, Bangladesh',
  'metadata': {'source_file': 'My Resume (1).pdf',
   'doc_index': 16,
   'page_label': '1',
   'creationdate': '2026-08-03T08:28:34+00:00',
   'total_pages': 2,
   'file_type': 'pdf',
   'creator': 'Chromium',
   'page': 0,
   'source': '..\\data\\pdf\\My Resume (1).pdf',
   'content_length': 555,
   'moddate': '2026-08-03T08:28:34+00:00',
   'producer': 'Skia/PDF m123'},
  'similarity_

In [14]:
rag_retriever.retrieve("The process of political development in Bangladesh")

Retrieving documents for query: 'The process of political development in Bangladesh'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.72it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_01edcc60_0',
  'content': 'Book Review Report \nBook Title - "More Than Meets the Eye: Essays on Bangladeshi Politics" \nBibliographic Information \nAuthor: Ali Riaz \nPublisher: University Press Limited (UPL), Dhaka \nYear of Publication: 2022 \nPages: 238 \nISBN: 9789845063777 \n \n \n1. Introduction : \n \nThe process of political development in  Bangladesh since independence is very complex. The \npolitical trajectory  consists of general election histories , institutional and civilian force and \namendments of the constitution. The book has giv en a concise  analytical insight through some \nthoughtful essays . Mr Riaz critically analyzed that the inner dynamics of politics  in Bangladesh  \nwere influential over political institutions, religious  institutes, civil- military relationship , \ndemocratic status and governance system. Mr Riaz depicted that numerous political reality keep \nhidden which can be understood only by analyzing historical and institutional  tre

something missing as like github

Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
# groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key = "GROQ_API_KEY"  # Replace with your actual API key

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [26]:
try:
    response = llm.invoke("Hello, how are you?")
    print(response.content)
except Exception as e:
    print(type(e).__name__)
    print(str(e))

Hello! I'm doing great, thank you for asking. How can I assist you today?


In [34]:
answer=rag_simple("what is Analysis of main theme?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'what is Analysis of main theme?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]


Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.


Enhanced RAG Pipeline Features

In [41]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("M.Sc. in Computer Science and Engineering", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'M.Sc. in Computer Science and Engineering'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: M.Sc. in Computer Science and Engineering, Islamic University Kushtia (Kushtia, Bangladesh), completed May 2023. Coursework included Machine Learning, Artificial Intelligence, Data Analysis, Natural Language Processing, Computer Graphics, and Advanced Database Management Systems, with research‑oriented projects and practical AI/software development experience.
Sources: [{'source': 'My Resume (1).pdf', 'page': 0, 'score': 0.14893299341201782, 'preview': 'Summary\nAI Engineer and Web Developer with expertise in machine learning, artiﬁcial intelligence, and full-stack development.\nExperienced in building intelligent systems and scalable web applications using Python, TensorFlow, React, Node.js, and\nWordPress. Passionate about solving real-world problem...'}, {'source': 'My Resume (1).pdf', 'page': 0, 'score': 0.14893299341201782, 'preview': 'Summary\nAI Engineer and Web Developer with expertise in 

In [45]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Relevance to public administration", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Relevance to public administration'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: The reply indicates that there is no applicable information or background related to the query. As a result, no further details can be provided.
History: {'question': 'Relevance to public administration', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The reply indicates that there is no applicable information or background related to the query. As a result, no further details can be provided.'}
